**Quantum Markov Decision Process: Qubit reuse through Dynamic Circuit (AER simulation)** 

Challenges
- Resetting Qubits: Qubits must be reset to a known state (e.g., |0⟩) after each use. This can be achieved through uncomputation or measurement followed by reinitialization.
- Error Propagation: In noisy quantum systems, repeated operations on the same qubits may amplify errors. Error correction or mitigation techniques might be required.
- Circuit Complexity: Designing such circuits may increase the complexity of the overall quantum circuit but saves qubit resources.

This code is for Dynamic QMDP Simulation 
1) Two-state transitions 
2) Two-state transitions with return

1) Two-state transitions Dynamic QMDP Simulation

In [ ]:
# importing packages
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister, transpile
from qiskit_aer.backends import AerSimulator
from qiskit.circuit.library import RYGate
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

from typing import List

In [ ]:
from typing import List

# functions
def reverse_function(x):
    return x[::-1]

def _state_to_latex_ket(
    data: List[complex], max_size: int = 12, prefix: str = "", decimals: int = 10 
    ) :
    num = int(np.log2(len(data)))

    def ket_name(i):
        return bin(i)[2:].zfill(num)

    data = np.around(data, decimals)
    nonzero_indices = np.where(data != 0)[0].tolist()
    latex_terms = data[nonzero_indices] ### changes here 

    nextState_prop_dict= {}
    nextState_bits_dict = {}
    counter = 0
    
    for idx, ket_idx in enumerate(nonzero_indices):
            term = latex_terms[idx]
            ket = ket_name(ket_idx)
            
            # nextState_prop_dict[counter] = np.abs(np.real(term))
            nextState_prop_dict[counter] = (np.real(term))
            nextState_bits_dict[counter] = ket
            counter +=1
            
    return nextState_prop_dict,nextState_bits_dict

In [ ]:
# Classical environment

class ClassicToyEnv:

    def __init__(self, MaxSteps=np.Inf):
        
        self.MaxSteps= MaxSteps # Maximum number of steps before ending an episode
        
        # ========================================
        # MDP definition
        self.nS= 4 # Number of MDP states
        self.nA= 2 # Number of MDP actions
        self.nR= 4 # Number of possible rewards

        # # Transition function
        self.T= np.zeros((self.nS, self.nA, self.nS))
        self.T[0, 0, 1]= 0.6
        self.T[0, 0, 2]= 0.4
        self.T[0, 1, 0]= 0.1 #here
        self.T[0, 1, 1]= 0.9 #here
        self.T[1, 0, 0]= 0.2 
        self.T[1, 0, 1]= 0.8
        self.T[1, 1, 2]= 0.7
        self.T[1, 1, 3]= 0.3
        self.T[2, 0, 0]= 0.3
        self.T[2, 0, 2]= 0.7
        self.T[2, 1, 1]= 0.2
        self.T[2, 1, 3]= 0.8
        self.T[3, 0, 2]= 0.6
        self.T[3, 0, 3]= 0.4
        self.T[3, 1, 3]= 1.0
        
        # Reward function
        self.R= np.array([10, -5, 1, -10])
        
        # Current state and step
        self.currentState= None
        self.currentStep= None
        
    
    # returns the number of states in the environment
    def numberOfStates(self):
        return self.nS
       
    # returns the number of states in the environment
    def numberOfActions(self):
        return self.nA
    
    # Returns the transition probability from state s and action a to state sp
    def transitionProb(self, s, a, sp):
        return self.T[s, a, sp]
    
    # Returns the reward value for the transition (s,a)->sp, i.e. r(s,a,sp)
    def rewardValue(self, s, a, sp):
        return self.R[sp]
    
    # Resets the environment to default state
    def reset(self):
        # Set default env state and step number
        self.currentState= 0
        self.currentStep= 0
        return int(self.currentState)

    def StoppingCriterionSatisfied(self):
        return self.currentStep >= self.MaxSteps

In [ ]:
# Quantum Environment for Two State Transitions

class QuantumToyEnv(ClassicToyEnv):
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.Inf):
        super().__init__(MaxSteps)
    
        # ========================================
        # Quantum MDP representation 
        self.nqS= 2 # Number of qubits for state representation= log2(nS)
        self.nqA= 1 # Number of qubits required for action representation= log2(nA)
        self.nqR= 2 # Number of qubits for reward representation= log2(nR)

        # Quantum Registers for MDP 2-state transition 
        self.qS= QuantumRegister(self.nqS, name='qState') # Quantum Register to store input environment states
        self.qSp= QuantumRegister(self.nqS, name='qNextState') # Quantum Register to store output environment states
        self.qA= QuantumRegister(self.nqA, name='qAction') # Quantum Register to store actions
        self.qR= QuantumRegister(self.nqR, name='qReward') # Quantum Register to store rewards
               
        # Classical Registers
        self.t0_cReg= ClassicalRegister(7, name='t0_cReg') # Classical Register to store measured timestep 0
        self.t1_cReg= ClassicalRegister(7, name='t1_cReg') # Classical Register to store measured timestep 1
        self.t2_cReg= ClassicalRegister(7, name='t2_cReg') # Classical Register to store measured timestep 2
        
        # Quantum Circuit to store a cycle of an MDP
        self.qc= QuantumCircuit(self.qS, self.qA, self.qSp, self.qR, self.t0_cReg, self.t1_cReg, self.t2_cReg)
        
        # ----------------------------------------------------------------------------------------------------------------
        # for t = 0 , Initial step      
        # Us implementation: Appends Us to self.qc
        self.__Us_0(self.qc, self.qS)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        """ Mid-ciruit Measurement at timestep 0 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t0_cReg[0], self.t0_cReg[1], self.t0_cReg[2], self.t0_cReg[3], self.t0_cReg[4], self.t0_cReg[5], self.t0_cReg[6]])
        
        self.qc.barrier()
        
        # ------------------------------------------------------------------------------------------------------------
        # for t = 1     
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 0 to t = 1. """       
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)
        
        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
       
         
        """ Mid-ciruit Measurement at timestep 1 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t1_cReg[0], self.t1_cReg[1], self.t1_cReg[2], self.t1_cReg[3], self.t1_cReg[4], self.t1_cReg[5], self.t1_cReg[6]])
        
        self.qc.barrier()
        
        # -----------------------------------------------------------------------------------------------------------------
        # for t = 2
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 1 to t = 2. """    
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)

        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)

        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)

        """ Mid-ciruit Measurement at timestep 1 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t2_cReg[0], self.t2_cReg[1], self.t2_cReg[2], self.t2_cReg[3], self.t2_cReg[4], self.t2_cReg[5], self.t2_cReg[6]])       

    # Implementation of the QSample encoding
    def __QSampleNextState(self, probs, qc, qS, qA, qSp, control_s, control_a, control_sp= '', currentQ=0):
        
        all_controls= control_s + control_a + control_sp
        
        # Find control qubits
        controlsS= []
        for i in range(len(qS)):
            controlsS.append(qS[i])
        controlsA= []
        for i in range(len(qA)):
            controlsA.append(qA[i])
        controlsSp= []
        for i in range(currentQ):
            controlsSp.append(qSp[i])
        
        # Base case: End of recursion
        if currentQ>=len(qSp):
            return
        
        else: # General case
        
            # compute prob(qS[currentQ | qS, qA, qS[:currentQ-1]])
            add_prob= np.sum(probs)
            if add_prob > 0:
                current_prob= np.sum(probs[len(probs)//2:])/add_prob
                angle= 2*np.arcsin(np.sqrt(current_prob))
                
                # Make custom controlled CRY gate 
                cry= RYGate(angle).control(num_ctrl_qubits=len(all_controls), 
                                             ctrl_state= all_controls)
                
                qc.append(cry, [*controlsSp, *controlsA, *controlsS, qSp[currentQ]])
        
            self.__QSampleNextState(probs[:len(probs)//2], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'0', currentQ=currentQ+1)
            self.__QSampleNextState(probs[len(probs)//2:], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'1', currentQ=currentQ+1)
    
    # Circuit to set input environment state  
    # for state in superposition
    def __Us_0(self, qc, qS):
        for i in range(len(qS)):
            qc.h(qS[i])

    def __Us(self, qc, qSp, qS):   
        for i in range(len(qS)):
           if i == 1:
             qc.cx(qSp[i],qS[i-1])
           else:
            qc.cx(qSp[i],qS[i+1])
       
    # Circuit to set input action    
    def __Ua(self, qc, qA):
        for i in range(len(qA)):
            qc.h(qA[i])
    
    # Circuit to implement transition function
    def __Ut(self, qc, qS, qA, qSp):
        
        for s in range(self.nS): 
            for a in range(self.nA):
                
                # Conditionals for input state s
                binS= bin(s)[2:][::-1]
                while len(binS)<self.nqS:
                    binS+= '0'
                binS= binS[::-1]
                
                # Conditionals for input action a
                binA= bin(a)[2:][::-1]
                while len(binA)<self.nqA:
                    binA+= '0'
                binA= binA[::-1]
        
                # Transition Function P(s,a)
                probs= self.T[s, a]
                self.__QSampleNextState(probs, self.qc, qS, qA, qSp, 
                        control_s=binS, 
                        control_a=binA)
    
    # Circuit to implement reward function
    def __Ur(self, qc, qSp, qR):
        for i in range(len(qSp)):
            qc.cx(qSp[i], qR[i])
          

In [ ]:
# Instantiate environment
qMDP_t2= QuantumToyEnv() 
qMDP_t2.qc.draw('mpl', fold=-1, initial_state=True) 

In [ ]:
# AER simulator

simulator = AerSimulator(method = 'statevector')
transpile_qMDP_t2 = transpile(qMDP_t2.qc, simulator)
results_qMDP_t2 = simulator.run(transpile_qMDP_t2, shots=1024, memory=True).result()
counts_qMDP_t2= results_qMDP_t2.get_counts()
print(len(counts_qMDP_t2))
print(counts_qMDP_t2)

In [ ]:
print ('counts_qMDP_t2:',counts_qMDP_t2) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # for t2
         t2 = value[:7]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[8:15]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[16:]
         t0_msb = reverse_qubit_order_msb(t0)
        #  return t2_msb+ " "+ t1_msb + " " + t0_msb
         return t2_msb+ t1_msb + t0_msb
     return value

# Create a new dictionary with modified values
modified_counts_qMDP_t2 = {correct_qubit_order(key): value for key, value in counts_qMDP_t2.items()}
print('modified_counts_qMDP_t2:',modified_counts_qMDP_t2) 

In [ ]:
len(modified_counts_qMDP_t2)

In [ ]:
# Get list of values from the dictionary
values = list(counts_qMDP_t2.values())

# Calculate the total sum manually
total = 0
for value in values:
    total += value

# Print the result
print("Total sum:", total)

Check with State vector

In [ ]:
# read saved Statevector

import os
import pickle

file_name = './save_QMDP_Static_Circuit/StateVector_QMDP Static Circuit_Two-State Transition.pkl'

if os.path.exists(file_name):
 print("...found existing State Vector! Reading it...")
 with open(file_name, "rb") as file:
      svector_3steps= pickle.load(file)

print(svector_3steps)

In [ ]:
prob_amplitudes, qOutputStates = _state_to_latex_ket(svector_3steps.data)
print('Probability amplitude and qsample trajectory:\n',prob_amplitudes,'\n',qOutputStates)

# sorting ascending order for amplitude values
prob_amplitudes_sorted = dict(sorted(prob_amplitudes.items(), key= lambda item: item[1]))
print('prob_amplitudes_sorted:',prob_amplitudes_sorted)

# top 5 values of amplitude
top_5_indices = np.argsort(list(prob_amplitudes.values()))[-5:]
print('top 5 indices:',top_5_indices)

# checking the magnitude of statevector is sum to '1'
sum = 0
for i in prob_amplitudes.values():
    i = abs(i)**2
    sum = sum + i
print('total sum:',sum)

print(len(qOutputStates))

In [ ]:
print ('qOutputStates:',qOutputStates) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # for t2
         t2 = value[:7]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[7:14]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[14:]
         t0_msb = reverse_qubit_order_msb(t0)
         return t2_msb+ t1_msb + t0_msb
     return value

# Create a new dictionary with modified values
modified_qOutputStates  = {key: correct_qubit_order(value) for key, value in qOutputStates.items()}
print('modified_qOutputStates:',modified_qOutputStates) ## Next state qubits and reward qubits are in correct order.

In [ ]:
# filter only the valid quantum samples

valid_samples = list(modified_qOutputStates.values()) # Next state qubits and reward qubits' orders are corrected.

filter_valid_modified_qMDP_t2= {key: value for key, value in modified_counts_qMDP_t2.items() if key in valid_samples}
print(f'Total QSample simulation: {len(filter_valid_modified_qMDP_t2)} samples.')
print('QSample simulation_correct qubit order:', filter_valid_modified_qMDP_t2)

valid_counts_sorted = dict(sorted(filter_valid_modified_qMDP_t2.items(), key= lambda item: item[0], reverse= False))
# valid_counts_sorted

In [ ]:
# read saved file # qubit order corrected

import os
import pickle
from qiskit.visualization import plot_histogram

file_name = './previous_static_QMDP_data/Reference 170 Trajectories without Return_Static QMDP.pkl' 

if os.path.exists(file_name):
 print("...found existing Classical Measurement! Reading it...")
 with open(file_name, "rb") as file:
       reference_170_trajectories = pickle.load(file) 

print(reference_170_trajectories)
print(len(reference_170_trajectories))

In [ ]:
eligible_states = list(reference_170_trajectories.keys())
len(eligible_states)

In [ ]:
filtered_counts= {key: value for key, value in modified_counts_qMDP_t2.items() if key in eligible_states}
print('filtered_classical_measurement:',len(filtered_counts))

modified_counts_qMDP_t2_sorted = dict(sorted(filtered_counts.items(), key= lambda item: item[0], reverse= False))

In [ ]:
modified_counts_qMDP_t2_sorted.keys()

Plot the Figures

In [ ]:
qStates = list(reference_170_trajectories.keys()) # Next state qubits and reward qubits are in correct order.
count = [modified_counts_qMDP_t2_sorted.get(key,0)for key in qStates] # Get corresponding values from 'filter_correct_trajectories_IBM_sorted', fill with 0 if missing

# Create a DataFrame 
df = pd.DataFrame({
     'Quantum States': qStates,
    'Count': count
})

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count').index  
top_10_values = df.loc[top_10_indices]
print('top_10_values:',top_10_values)

top_colors  = ['#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000']
df['Color'] = ('#0072B2')

# make a plot
fig, ax = plt.subplots(figsize=(20, 7))

# Define the width of the bars 
bar_width = 0.6 # pdf

# Create the bar chart
bars = ax.bar(df['Quantum States'], df['Count'], color=df['Color'], width=bar_width)

# Set the labels and title
ax.set_xlabel('Quantum Trajectory', fontsize = 12, fontfamily ='Arial', labelpad= 12)
ax.set_ylabel('Total Count', fontsize = 12, fontfamily ='Arial', labelpad= 12)
ax.set_title('Quantum Trajectories in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps via Dynamic Circuit', fontsize = 18, fontfamily ='Arial')


# Rotate x-axis labels for better readability
plt.xticks(rotation=90, fontsize = 8)
plt.yticks(rotation=0, fontsize = 12, fontfamily='Arial')

# Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height+ 0.1 , f'{height:.0f}', ha='center', va='bottom', fontsize ='medium', rotation = 0, fontfamily ='Arial')


# Add space on the left and right
left_padding = -2.2  # Add more space to the left
right_padding = len(df['Quantum States']) - 0.5 + 2  # Add more space to the right
ax.set_xlim(left_padding, right_padding)

# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
###### x axis label changed 

# Quantum States and Counts
qStates = list(reference_170_trajectories.keys()) # Next state qubits and reward qubits are in correct order.
count = [modified_counts_qMDP_t2_sorted.get(key,0)for key in qStates] # Get corresponding values from 'filter_correct_trajectories_IBM_sorted', fill with 0 if missing

# Create a DataFrame
df = pd.DataFrame({
    'Quantum States': qStates,
    'Count': count
})

# Add an index column for sequential numbering
df['Index'] = range(1, len(df) + 1)

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count').index  
top_10_values = df.loc[top_10_indices]
# print('top_10_values:',top_10_values)

top_colors  = ['#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000']

# Assign colors (optional for customization)
df['Color'] = '#0072B2'

# make a plot
fig, ax = plt.subplots(figsize=(20, 7))

# Define the width of the bars 
bar_width = 0.8

# Create the bar chart
bars = ax.bar(df['Index'], df['Count'], color=df['Color'], width=bar_width)

# Set labels and title
ax.set_xlabel('Quantum Trajectory', fontsize=16, fontfamily='Arial', labelpad=12)
ax.set_ylabel('Total Count', fontsize=16, fontfamily='Arial', labelpad=12)
ax.set_title('Quantum Trajectories in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps via Dynamic Circuit', fontsize = 18, fontfamily ='Arial')

# Set the x-axis ticks with T+ format
t_plus_labels = [f'T-{i}' for i in df['Index']]
plt.xticks(df['Index'], t_plus_labels, rotation=90, fontsize=9.5, fontfamily='Arial')
plt.yticks(rotation=0, fontsize=14, fontfamily='Arial')

###Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.05, f'{height:.0f}', ha='center', va='bottom', fontsize='medium', rotation=0, fontfamily='Arial')
    
# Add space on the left and right
left_padding = -0.5  # Add more space to the left
right_padding = len(df['Index']) - 0.5 + 2  # Add more space to the right
ax.set_xlim(left_padding, right_padding)

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
# Quantum States and Counts
qStates = list(reference_170_trajectories.keys()) # Next state qubits and reward qubits are in correct order.
count = [modified_counts_qMDP_t2_sorted.get(key,0)for key in qStates] # Get corresponding values from 'filter_correct_trajectories_IBM_sorted', fill with 0 if missing

# Create a DataFrame
df = pd.DataFrame({
    'Quantum States': qStates,
    'Count': count
})

df.index = range(1, len(df) + 1) # setting data frame index starting from 1 not 0
# Add an index column for sequential numbering
df['Index'] = range(1, len(df) + 1)

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count').index  
top_10_values = df.loc[top_10_indices]
# print('top_10_values:',top_10_values)

top_colors  = ['#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000']

# Assign colors (optional for customization)
df['Color'] = '#0072B2'

# make a plot
# fig, ax = plt.subplots(figsize=(20, 7))
fig, ax = plt.subplots(figsize=(20, 5.5))

# Define the width of the bars 
# bar_width = 0.8
bar_width = 0.6 # pdf

# Create the bar chart
bars = ax.bar(df['Index'], df['Count'], color=df['Color'], width=bar_width)

# Set labels and title
ax.set_xlabel('Quantum Trajectory', fontsize=28, fontfamily='Arial', labelpad=3)
ax.set_ylabel('Total Count', fontsize=28, fontfamily='Arial', labelpad=2)
ax.set_title('Quantum Trajectories in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps via Dynamic Circuit', fontsize = 18, fontfamily ='Arial')

# Generate x-axis tick labels: every 10th tick + include the last label
tick_interval = 10
max_index = df['Index'].max()
selected_indices = list(range(1, max_index + 1, tick_interval))  # Every 10th tick
if max_index not in selected_indices:  # Include last index if not already there
    selected_indices.append(max_index)

# Generate labels for the selected indices
t_plus_labels = [f'T-{i}' for i in selected_indices]

# Update x-axis ticks and labels
plt.xticks(selected_indices, t_plus_labels, rotation=90, fontsize=22, fontfamily='Arial')

ax.set_yticks(range(0, int(df['Count'].max()) + 1, 10))  # Tick range: 0 to max, step = 5
plt.yticks(rotation=0, fontsize=24, fontfamily='Arial')

###Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.5, f'{height:.0f}', ha='center', va='bottom', fontsize='xx-large', rotation=0, fontfamily='Arial')
    
# Add space on the left and right
left_padding = -0.1  # Add more space to the left
right_padding = len(df['Index']) - 0.5 + 1.3  # Add more space to the right
ax.set_xlim(left_padding, right_padding)

# Adjust Y-Axis Limit to Add Padding
y_max = max(df['Count'])  # Maximum value in the dataset
ax.set_ylim(0, y_max + 3)  # Add some padding above the tallest bar

plt.tight_layout()

plt.show()


----

2) Two-state transitions with return

In [ ]:
# importing packages
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister, transpile
from qiskit_aer.backends import AerSimulator
from qiskit.circuit.library import RYGate
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import DraperQFTAdder

import pandas as pd

import numpy as np
import matplotlib.pyplot as plt

from typing import List

In [ ]:
from typing import List

# functions
def reverse_function(x):
    return x[::-1]

def _state_to_latex_ket(
    data: List[complex], max_size: int = 12, prefix: str = "", decimals: int = 10 
    ) :
    num = int(np.log2(len(data)))

    def ket_name(i):
        return bin(i)[2:].zfill(num)

    data = np.around(data, decimals)
    nonzero_indices = np.where(data != 0)[0].tolist()
    latex_terms = data[nonzero_indices] ### changes here 

    nextState_prop_dict= {}
    nextState_bits_dict = {}
    counter = 0
    
    for idx, ket_idx in enumerate(nonzero_indices):
            term = latex_terms[idx]
            ket = ket_name(ket_idx)
            
            # nextState_prop_dict[counter] = np.abs(np.real(term))
            nextState_prop_dict[counter] = (np.real(term))
            nextState_bits_dict[counter] = ket
            counter +=1
            
    return nextState_prop_dict,nextState_bits_dict

In [ ]:
# Classical environment

class ClassicToyEnv:

    def __init__(self, MaxSteps=np.inf):
        
        self.MaxSteps= MaxSteps # Maximum number of steps before ending an episode
        
        # ========================================
        # MDP definition
        self.nS= 4 # Number of MDP states
        self.nA= 2 # Number of MDP actions
        self.nR= 4 # Number of possible rewards

        # # Transition function
        self.T= np.zeros((self.nS, self.nA, self.nS))
        self.T[0, 0, 1]= 0.6
        self.T[0, 0, 2]= 0.4
        self.T[0, 1, 0]= 0.1 #here
        self.T[0, 1, 1]= 0.9 #here
        self.T[1, 0, 0]= 0.2 
        self.T[1, 0, 1]= 0.8
        self.T[1, 1, 2]= 0.7
        self.T[1, 1, 3]= 0.3
        self.T[2, 0, 0]= 0.3
        self.T[2, 0, 2]= 0.7
        self.T[2, 1, 1]= 0.2
        self.T[2, 1, 3]= 0.8
        self.T[3, 0, 2]= 0.6
        self.T[3, 0, 3]= 0.4
        self.T[3, 1, 3]= 1.0
        
        # Reward function
        self.R= np.array([10, -5, 1, -10])
        
        # Current state and step
        self.currentState= None
        self.currentStep= None
        
    
    # returns the number of states in the environment
    def numberOfStates(self):
        return self.nS
       
    # returns the number of states in the environment
    def numberOfActions(self):
        return self.nA
    
    # Returns the transition probability from state s and action a to state sp
    def transitionProb(self, s, a, sp):
        return self.T[s, a, sp]
    
    # Returns the reward value for the transition (s,a)->sp, i.e. r(s,a,sp)
    def rewardValue(self, s, a, sp):
        return self.R[sp]
    
    # Resets the environment to default state
    # Returns the current env. state
    def reset(self):
        # Set default env state and step number
        self.currentState= 0
        self.currentStep= 0
        return int(self.currentState)

    def StoppingCriterionSatisfied(self):
        return self.currentStep >= self.MaxSteps

In [ ]:
# Quantum Environment for Two State Transitions with Return
# Storing Reward qubits into Ancillary qubits before mid-circuit measurement

class QuantumToyEnv(ClassicToyEnv):
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.inf):
        super().__init__(MaxSteps)
    
        # ========================================
        # Quantum MDP representation 
        self.nqS= 2 # Number of qubits for state representation= log2(nS)
        self.nqA= 1 # Number of qubits required for action representation= log2(nA)
        self.nqR= 2 # Number of qubits for reward representation= log2(nR)
        self.nqRt = 4 # Number of qubits for return
        self.nqAnci = 2 # Number of ancillary qubits

        # Quantum Registers for MDP 2-state transition 
        self.qS= QuantumRegister(self.nqS, name='qState') # Quantum Register to store input environment states
        self.qSp= QuantumRegister(self.nqS, name='qNextState') # Quantum Register to store output environment states
        self.qA= QuantumRegister(self.nqA, name='qAction') # Quantum Register to store actions
        self.qR= QuantumRegister(self.nqR, name='qReward') # Quantum Register to store rewards
        
        self.t0_qAnci = QuantumRegister(self.nqAnci, name = 't0_qAnci') # ancillary qubits to store the timestep 0 reward
        self.t1_qAnci = QuantumRegister(self.nqAnci, name = 't1_qAnci') # ancillary qubits to store the timestep 1 reward
        
        # Quantum Registers for Return
        self.qRt= QuantumRegister(self.nqRt, name='qRt') # Quantum Register to store return
        
        # Classical Registers
        self.t0_cReg= ClassicalRegister(7, name='t0_cReg') # Classical Register to store measured timestep 0
        self.t1_cReg= ClassicalRegister(7, name='t1_cReg') # Classical Register to store measured timestep 1
        self.t2_cReg= ClassicalRegister(7, name='t2_cReg') # Classical Register to store measured timestep 2
        self.cRt= ClassicalRegister(4, name='cR') # Classical Register to store measured return
        
        # Quantum Circuit to store a cycle of an MDP
        self.qc= QuantumCircuit(self.qS, self.qA, self.qSp, self.qR, self.t0_qAnci,  self.t1_qAnci, self.qRt, 
                                           self.t0_cReg, self.t1_cReg, self.t2_cReg, self.cRt)
        
        # ----------------------------------------------------------------------------------------------------------------
        # for t = 0 , Initial step      
        # Us implementation: Appends Us to self.qc
        self.__Us_0(self.qc, self.qS)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t0_qAnci) 
        
        self.qc.barrier()
        
        """ Mid-ciruit Measurement at timestep 0 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t0_cReg[0], self.t0_cReg[1], self.t0_cReg[2], self.t0_cReg[3], self.t0_cReg[4], self.t0_cReg[5], self.t0_cReg[6]])
        
        self.qc.barrier()
              
        # ------------------------------------------------------------------------------------------------------------
        # for t = 1     
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 0 to t = 1. """       
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)
        
        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t1_qAnci) 
        
        self.qc.barrier()
         
        """ Mid-ciruit Measurement at timestep 1 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t1_cReg[0], self.t1_cReg[1], self.t1_cReg[2], self.t1_cReg[3], self.t1_cReg[4], self.t1_cReg[5], self.t1_cReg[6]])
        
        self.qc.barrier()
               
        # -----------------------------------------------------------------------------------------------------------------
        # for t = 2
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 1 to t = 2. """    
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)

        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)

        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)

        self.qc.barrier()
        
        # -----------------------------------------------------------------------------------------------------------------
        # Return
        self.__UReturn(self.qc, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt)
        
        """ Mid-ciruit Measurement at timestep 2 """
        # ====> Mid-ciruit Measurement for timestep 0 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.t2_cReg[0], self.t2_cReg[1], self.t2_cReg[2], self.t2_cReg[3], self.t2_cReg[4], self.t2_cReg[5], self.t2_cReg[6]])       
       
        self.qc.barrier()
        
        self.qc.measure([self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3]], 
                               [self.cRt[0], self.cRt[1], self.cRt[2], self.cRt[3]])     
        
    # Implementation of the QSample encoding
    def __QSampleNextState(self, probs, qc, qS, qA, qSp, control_s, control_a, control_sp= '', currentQ=0):
        
        all_controls= control_s + control_a + control_sp
        
        # Find control qubits
        controlsS= []
        for i in range(len(qS)):
            controlsS.append(qS[i])
        controlsA= []
        for i in range(len(qA)):
            controlsA.append(qA[i])
        controlsSp= []
        for i in range(currentQ):
            controlsSp.append(qSp[i])
        
        # Base case: End of recursion
        if currentQ>=len(qSp):
            return
        
        else: # General case
        
            # compute prob(qS[currentQ | qS, qA, qS[:currentQ-1]])
            add_prob= np.sum(probs)
            if add_prob > 0:
                current_prob= np.sum(probs[len(probs)//2:])/add_prob
                angle= 2*np.arcsin(np.sqrt(current_prob))
                
                # Make custom controlled CRY gate 
                cry= RYGate(angle).control(num_ctrl_qubits=len(all_controls), 
                                             ctrl_state= all_controls)
                
                qc.append(cry, [*controlsSp, *controlsA, *controlsS, qSp[currentQ]])
        
            self.__QSampleNextState(probs[:len(probs)//2], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'0', currentQ=currentQ+1)
            self.__QSampleNextState(probs[len(probs)//2:], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'1', currentQ=currentQ+1)
    

    # Circuit to set input environment state  
    # for state in superposition
    def __Us_0(self, qc, qS):
        for i in range(len(qS)):
            qc.h(qS[i])

    def __Us(self, qc, qSp, qS):   
        for i in range(len(qS)):
           if i == 1:
             qc.cx(qSp[i],qS[i-1])
           else:
            qc.cx(qSp[i],qS[i+1])
       
    # Circuit to set input action    
    def __Ua(self, qc, qA):
        for i in range(len(qA)):
            qc.h(qA[i])
    
    # Circuit to implement transition function
    def __Ut(self, qc, qS, qA, qSp):
        
        for s in range(self.nS): 
            for a in range(self.nA):
                
                # Conditionals for input state s
                binS= bin(s)[2:][::-1]
                while len(binS)<self.nqS:
                    binS+= '0'
                binS= binS[::-1]
                
                # Conditionals for input action a
                binA= bin(a)[2:][::-1]
                while len(binA)<self.nqA:
                    binA+= '0'
                binA= binA[::-1]
        
                # Transition Function P(s,a)
                probs= self.T[s, a]
                self.__QSampleNextState(probs, self.qc, qS, qA, qSp, 
                        control_s=binS, 
                        control_a=binA)

    # Circuit to implement reward function
    def __Ur(self, qc, qSp, qR):
        for i in range(len(qSp)):
            qc.cx(qSp[i], qR[i])

    # Circuit to store the reward at the ancillary qubits
    def __Uanci(self, qc, qR, qAnci):
        for i in range(len(qR)):
            qc.cx(qR[i], qAnci[i])       


    # Circuit to store the reward at the ancillary qubits
    def __UReturn(self, qc, t0_qReward, t1_qReward, qReward, qReturn):
      ## For Return qubit[0]
      qc.cx(t0_qReward[1], qReturn[0])
      qc.cx(t1_qReward[1], qReturn[0])
      qc.cx(qReward[1], qReturn[0])
      
      qc.barrier()

      ## For Return qubit[1]
      qc.cx(t0_qReward[0], qReturn[1])
      qc.cx(t1_qReward[0], qReturn[1])
      qc.cx(qReward[0], qReturn[1])
      # For carry-in part
      qc.ccx(t0_qReward[1], t1_qReward[1], qReturn[1])
      qc.ccx(t0_qReward[1], qReward[1], qReturn[1])
      qc.ccx(t1_qReward[1], qReward[1], qReturn[1])
      qc.barrier()
      
      # For Return qubit [2]
      # when no carry-in bit from LSB
      qc.ccx(t0_qReward[0],  t1_qReward[0], qReturn[2])
      qc.ccx(t0_qReward[0], qReward[0],qReturn[2])
      qc.ccx(t1_qReward[0], qReward[0] ,qReturn[2])

      # when carry-in bit from LSB
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      ## carry-in and single one qubit
      qc.mcx([t0_qReward[1], t1_qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1],  t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t1_qReward[1], qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])

      qc.barrier()

      ### for  Return qubit [3] (carry-out qubit)
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[3]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.barrier()


In [ ]:
# Instantiate environment
qMDP_return= QuantumToyEnv()
qMDP_return.qc.draw('mpl', fold=-1, initial_state=True) # 'fold=-1' ensures no line breaks

In [ ]:
# AER simulator

simulator = AerSimulator(method = 'statevector')
transpile_qMDP_return = transpile(qMDP_return.qc, simulator)
results_qMDP_return = simulator.run(transpile_qMDP_return, shots=1024, memory=True).result()
counts_qMDP_return= results_qMDP_return.get_counts()
print(len(counts_qMDP_return))
print(counts_qMDP_return)

In [ ]:
len(counts_qMDP_return)

In [ ]:
print ('counts_qMDP_return:',counts_qMDP_return) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # return
         rt = value[:4] 
         # for t2
         t2 = value[5:12]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[13:20]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[21:]
         t0_msb = reverse_qubit_order_msb(t0)
        #  return rt+ " " + t2_msb+ " "+ t1_msb + " " + t0_msb
         return rt+ t2_msb+ t1_msb + t0_msb
     return value

# Create a new dictionary with modified values
modified_counts_qMDP_return = {correct_qubit_order(key): value for key, value in counts_qMDP_return.items()}
print('modified_counts_qMDP_return:',modified_counts_qMDP_return) 

In [ ]:
len(modified_counts_qMDP_return)

In [ ]:
# Get list of values from the dictionary
values = list(modified_counts_qMDP_return.values())

# Calculate the total sum manually
total = 0
for value in values:
    total += value

# Print the result
print("Total sum:", total)

Check with State Vetor

In [ ]:
# read saved Statevector

import os
import pickle

file_name = './save_QMDP_Static_Circuit/Two State Transitions with Return Statevectors.pkl'

if os.path.exists(file_name):
 print("...found existing State Vector! Reading it...")
 with open(file_name, "rb") as file:
      svector_3steps_return= pickle.load(file)

print(svector_3steps_return)

In [ ]:
prob_amplitudes, qOutputStates = _state_to_latex_ket(svector_3steps_return.data)
print('Probability amplitude and qsample trajectory:\n',prob_amplitudes,'\n',qOutputStates)

# sorting ascending order for amplitude values
prob_amplitudes_sorted = dict(sorted(prob_amplitudes.items(), key= lambda item: item[1]))
print('prob_amplitudes_sorted:',prob_amplitudes_sorted)

# top 5 values of amplitude
top_5_indices = np.argsort(list(prob_amplitudes.values()))[-5:]
print('top 5 indices:',top_5_indices)

# checking the magnitude of statevector is sum to '1'
sum = 0
for i in prob_amplitudes.values():
    i = abs(i)**2
    sum = sum + i
print('total sum:',sum)

print(len(qOutputStates))

In [ ]:
print ('qOutputStates:',qOutputStates) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # return
         rt = value[:4] 
         # for t2
         t2 = value[4:11]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[11:18]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[18:]
         t0_msb = reverse_qubit_order_msb(t0)
         return rt+t2_msb+ t1_msb + t0_msb
     return value

# Create a new dictionary with modified values
modified_qOutputStates  = {key: correct_qubit_order(value) for key, value in qOutputStates.items()}
print('modified_qOutputStates:',modified_qOutputStates) ## Next state qubits and reward qubits are in correct order.

In [ ]:
# filter only the valid quantum samples

valid_samples = list(modified_qOutputStates.values()) 
filter_valid_modified_qMDP_return= {key: value for key, value in modified_counts_qMDP_return.items() if key in valid_samples}
print(f'Total QSample simulation: {len(filter_valid_modified_qMDP_return)} samples.')
print('QSample simulation_correct qubit order:', filter_valid_modified_qMDP_return)

valid_counts_sorted = dict(sorted(filter_valid_modified_qMDP_return.items(), key= lambda item: item[0], reverse= False))

In [ ]:
# read saved file: the reference 170 trajectories 

import os
import pickle
from qiskit.visualization import plot_histogram

file_name = './previous_static_QMDP_data/Classical Counts for Two State Transitions with Return.pkl'

if os.path.exists(file_name):
 print("...found existing Classical Measurement! Reading it...")
 with open(file_name, "rb") as file:
       counts_mdp = pickle.load(file)

print(counts_mdp)
plot_histogram(counts_mdp, title= 'Reference Trajectories Static QMDP: Two-State Transitions with return')

In [ ]:
print ('counts_mdp:',counts_mdp) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # return
         rt = value[:4] 
         # for t2
         t2 = value[4:11]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[11:18]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[18:]
         t0_msb = reverse_qubit_order_msb(t0)
         return rt+t2_msb+ t1_msb+ t0_msb
        #  return rt+ " " + t2_msb+ " "+ t1_msb + " " + t0_msb
     return value

# Create a new dictionary with modified values
modified_counts_mdp = {correct_qubit_order(key): value for key, value in counts_mdp.items()}
print('modified_counts_mdp:',modified_counts_mdp) 

In [ ]:
modified_counts_mdp_sorted = dict(sorted(modified_counts_mdp.items(), key= lambda item: item[0], reverse= False))
modified_counts_mdp_sorted

reference_170_trajectories_with_return = list(modified_counts_mdp_sorted.keys())
len(reference_170_trajectories_with_return)

In [ ]:
filtered_counts_qMDP_return= {key: value for key, value in modified_counts_qMDP_return.items() if key in reference_170_trajectories_with_return}
print('filtered_classical_measurement:',len(filtered_counts_qMDP_return))

filtered_counts_qMDP_return_sorted = dict(sorted(filtered_counts_qMDP_return.items(), key= lambda item: item[0], reverse= False))
filtered_counts_qMDP_return_sorted

Plotting

In [ ]:
qStates = reference_170_trajectories_with_return
count = [filtered_counts_qMDP_return_sorted.get(key,0)for key in qStates]

# Create a DataFrame 
df = pd.DataFrame({
     'Quantum States': qStates,
    'Count': count
})

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count').index  
top_10_values = df.loc[top_10_indices]
print('top_10_values:',top_10_values)

top_colors  = ['#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000']
df['Color'] = ('#0072B2')

# make a plot
fig, ax = plt.subplots(figsize=(20, 7))

# Define the width of the bars 
bar_width = 0.6 # pdf

# Create the bar chart
bars = ax.bar(df['Quantum States'], df['Count'], color=df['Color'], width=bar_width)

# Set the labels and title
ax.set_xlabel('Quantum Trajectory', fontsize = 12, fontfamily ='Arial', labelpad= 12)
ax.set_ylabel('Total Count', fontsize = 12, fontfamily ='Arial', labelpad= 12)
ax.set_title('Quantum Trajectories in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps via Dynamic Circuit', fontsize = 18, fontfamily ='Arial')


# Rotate x-axis labels for better readability
plt.xticks(rotation=90, fontsize = 8)
plt.yticks(rotation=0, fontsize = 12, fontfamily='Arial')

# Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height+ 0.1 , f'{height:.0f}', ha='center', va='bottom', fontsize ='medium', rotation = 0, fontfamily ='Arial')

# Add space on the left and right
left_padding = -2.2  # Add more space to the left
right_padding = len(df['Quantum States']) - 0.5 + 2  # Add more space to the right
ax.set_xlim(left_padding, right_padding)

# Show the plot
plt.tight_layout()

plt.show()

In [ ]:
qStates = reference_170_trajectories_with_return 
count = [filtered_counts_qMDP_return_sorted.get(key,0)for key in qStates] 

# Create a DataFrame
df = pd.DataFrame({
    'Quantum States': qStates,
    'Count': count
})

# Add an index column for sequential numbering
df['Index'] = range(1, len(df) + 1)

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count').index  
top_10_values = df.loc[top_10_indices]

top_colors  = ['#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000']
# Assign colors (optional for customization)
df['Color'] = '#0072B2'

# make a plot
fig, ax = plt.subplots(figsize=(20, 7))

# Define the width of the bars 
bar_width = 0.8

# Create the bar chart
bars = ax.bar(df['Index'], df['Count'], color=df['Color'], width=bar_width)

# Set labels and title
ax.set_xlabel('Quantum Trajectory', fontsize=16, fontfamily='Arial', labelpad=12)
ax.set_ylabel('Total Count', fontsize=16, fontfamily='Arial', labelpad=12)
ax.set_title('Quantum Trajectories in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps via Dynamic Circuit', fontsize = 18, fontfamily ='Arial')

# Set the x-axis ticks with T+ format
t_plus_labels = [f'T-{i}' for i in df['Index']]
plt.xticks(df['Index'], t_plus_labels, rotation=90, fontsize=9.5, fontfamily='Arial')
plt.yticks(rotation=0, fontsize=14, fontfamily='Arial')

###Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.1, f'{height:.0f}', ha='center', va='bottom', fontsize='medium', rotation=0, fontfamily='Arial')
    
# Add space on the left and right
left_padding = -0.5  # Add more space to the left
right_padding = len(df['Index']) - 0.5 + 2  # Add more space to the right
ax.set_xlim(left_padding, right_padding)

# Show the plot
plt.tight_layout()

plt.show()


In [ ]:
qStates = reference_170_trajectories_with_return 
count = [filtered_counts_qMDP_return_sorted.get(key,0)for key in qStates] 
# Create a DataFrame
df = pd.DataFrame({
    'Quantum States': qStates,
    'Count': count
})

df.index = range(1, len(df) + 1) # setting data frame index starting from 1 not 0
# Add an index column for sequential numbering
df['Index'] = range(1, len(df) + 1)

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count').index  
top_10_values = df.loc[top_10_indices]

top_colors  = ['#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000']

# Assign colors (optional for customization)
df['Color'] = '#0072B2'

# make a plot
fig, ax = plt.subplots(figsize=(20, 5.5))

# Define the width of the bars 
bar_width = 0.6 # pdf

# Create the bar chart
bars = ax.bar(df['Index'], df['Count'], color=df['Color'], width=bar_width)

# Set labels and title
ax.set_xlabel('Quantum Trajectory', fontsize=28, fontfamily='Arial', labelpad=3)
ax.set_ylabel('Total Count', fontsize=28, fontfamily='Arial', labelpad=2)
ax.set_title('Quantum Trajectories in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps via Dynamic Circuit', fontsize = 18, fontfamily ='Arial')


# Generate x-axis tick labels: every 10th tick + include the last label
tick_interval = 10
max_index = df['Index'].max()
selected_indices = list(range(1, max_index + 1, tick_interval))  # Every 10th tick
if max_index not in selected_indices:  # Include last index if not already there
    selected_indices.append(max_index)

# Generate labels for the selected indices
t_plus_labels = [f'T-{i}' for i in selected_indices]

# Update x-axis ticks and labels
plt.xticks(selected_indices, t_plus_labels, rotation=90, fontsize=22, fontfamily='Arial')

ax.set_yticks(range(0, int(df['Count'].max()) + 1, 10))  # Tick range: 0 to max, step = 5
plt.yticks(rotation=0, fontsize=24, fontfamily='Arial')

###Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.5, f'{height:.0f}', ha='center', va='bottom', fontsize='x-large', rotation=0, fontfamily='Arial')
    
# Add space on the left and right
left_padding = -0.1  # Add more space to the left
right_padding = len(df['Index']) - 0.5 + 1.3  # Add more space to the right
ax.set_xlim(left_padding, right_padding)

# Adjust Y-Axis Limit to Add Padding
y_max = max(df['Count'])  # Maximum value in the dataset
ax.set_ylim(0, y_max + 3)  # Add some padding above the tallest bar

plt.tight_layout()

plt.show()


In [ ]:
qStates = reference_170_trajectories_with_return 
count = [filtered_counts_qMDP_return_sorted.get(key,0)for key in qStates] 

# Create a DataFrame
df = pd.DataFrame({
    'Quantum States': qStates,
    'Count': count
})

df.index = range(1, len(df) + 1) # setting data frame index starting from 1 not 0
# Add an index column for sequential numbering
df['Index'] = range(1, len(df) + 1)

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count').index  
top_10_values = df.loc[top_10_indices]
# print('top_10_values:',top_10_values)

top_colors  = ['#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000','#ff0000']

# Assign colors (optional for customization)
df['Color'] = '#0072B2'

# make a plot
# fig, ax = plt.subplots(figsize=(20, 7))
fig, ax = plt.subplots(figsize=(20, 5.5))

# Define the width of the bars 
# bar_width = 0.8
bar_width = 0.6 # pdf

# Create the bar chart
bars = ax.bar(df['Index'], df['Count'], color=df['Color'], width=bar_width)

# Set labels and title
ax.set_xlabel('Quantum Trajectory', fontsize=28, fontfamily='Arial', labelpad=3)
ax.set_ylabel('Total Count', fontsize=28, fontfamily='Arial', labelpad=2)
ax.set_title('Quantum Trajectories in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps via Dynamic Circuit', fontsize = 18, fontfamily ='Arial')


# Generate x-axis tick labels: every 10th tick + include the last label
tick_interval = 10
max_index = df['Index'].max()
selected_indices = list(range(1, max_index + 1, tick_interval))  # Every 10th tick
if max_index not in selected_indices:  # Include last index if not already there
    selected_indices.append(max_index)

# Generate labels for the selected indices
t_plus_labels = [f'T-{i}' for i in selected_indices]

# Update x-axis ticks and labels
plt.xticks(selected_indices, t_plus_labels, rotation=90, fontsize=22, fontfamily='Arial')

ax.set_yticks(range(0, int(df['Count'].max()) + 1, 10))  # Tick range: 0 to max, step = 5
plt.yticks(rotation=0, fontsize=8, fontfamily='Arial')

###Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.5, f'{height:.0f}', ha='center', va='bottom', fontsize='large', rotation=0, fontfamily='Arial')
    
# Add space on the left and right
left_padding = -0.1  # Add more space to the left
right_padding = len(df['Index']) - 0.5 + 1.3  # Add more space to the right
ax.set_xlim(left_padding, right_padding)

# Adjust Y-Axis Limit to Add Padding
y_max = max(df['Count'])  # Maximum value in the dataset
ax.set_ylim(0, y_max + 3)  # Add some padding above the tallest bar

plt.tight_layout()

plt.show()


Plot the figures for Trajectories vs visited state for each Timestep

In [ ]:
modified_counts_qMDP_return_sorted = dict(sorted(filtered_counts_qMDP_return_sorted.items(), key= lambda item: item[0], reverse= False))
modified_counts_qMDP_return_sorted

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# --- take the dictionary items (unique trajectories) ---
items = list(modified_counts_qMDP_return_sorted.items())[:]
print(items)

def parse_traj(key: str):
    parts = key.split()
    ret = parts[0]                          # keep return as string
    split_2, split_3, split_4 = parts[1], parts[2], parts[3]
    t0 = int(split_4[-2:])                  # t0 must come from 4th sequence
    t1 = int(split_3[-2:])                  # t1 must come from 3h sequence
    t2 = int(split_2[-2:])                  # t2 must come from 2th sequence
    return ret, [t0, t1, t2]

# Build one column per trajectory
states, returns = [], []
for key, count in items:
    ret, timesteps_states = parse_traj(key)
    states.append(timesteps_states)
    returns.append(ret)

# 3 x N array
states_arr = np.array(states).T

# convert binary-like ints to decimal
decimal_states_arr = np.array([[int(str(v), 2) for v in row] for row in states_arr])

# group consecutive same-returns (assumes items are already sorted by return)
unique_returns, groups = [], []
for i, r in enumerate(returns):
    if not unique_returns or r != unique_returns[-1]:
        unique_returns.append(r); groups.append([])
    groups[-1].append(i)

# create plot
fig, ax = plt.subplots(figsize=(20, 5.5))

# vertical separators between return groups
for i in range(1, len(groups)):
    boundary = groups[i][0] - 0.5
    ax.axvline(boundary, color='red', linestyle='--', linewidth=3)

# ===== CUSTOM COLORS PER STATE (s0..s3) =====
# define discrete colors and use them in imshow
state_colors = {
    0: "#3f1d5a",   # s0
    1: "#1f77b4",   # s1
    2: "#2ca02c",   # s2
    3: "#ffcc00",     # s3
}

states_present = sorted(np.unique(decimal_states_arr))
cmap = ListedColormap([state_colors[s] for s in states_present])   
bounds = [s - 0.5 for s in states_present] + [states_present[-1] + 0.5]
norm = BoundaryNorm(bounds, cmap.N)                               

# heatmap (use discrete cmap/norm)
im = ax.imshow(decimal_states_arr, aspect='auto', origin='lower', cmap=cmap, norm=norm)                              

# ---- X axis: label every 10 trajectories AND include the last index ----
n_cols = decimal_states_arr.shape[1]
major_idx = list(range(0, n_cols, 10))
if (n_cols - 1) not in major_idx:
    major_idx.append(n_cols - 1)

ax.set_xticks(np.arange(n_cols), minor=True)
ax.tick_params(axis='x', which='minor', length=2, labelsize=0)

ax.set_xticks(major_idx)
ax.set_xticklabels([f"T-{i+1}" for i in major_idx],  rotation=90, fontsize=20, fontfamily='Arial')

# Y axis
ax.set_yticks(range(decimal_states_arr.shape[0]))
ax.set_yticklabels([f"t={i}" for i in range(decimal_states_arr.shape[0])], fontsize=22, fontfamily='Arial')

# Secondary x-axis for group return values only
secax = ax.secondary_xaxis('top')
group_centers = [(g[0] + g[-1]) / 2 for g in groups]
secax.set_xticks(group_centers)
secax.set_xticklabels([str(unique_returns[i]) for i in range(len(groups))], fontsize=20, fontfamily='Arial')
secax.tick_params(which='minor', bottom=False, top=False)
secax.tick_params(axis='x', pad=8)        

# labels
ax.set_xlabel('Quantum Trajectories', fontsize=28, fontfamily='Arial', labelpad=10)
ax.set_ylabel('Timesteps', fontsize=28, fontfamily='Arial', labelpad=4)
secax.set_xlabel("Return", fontsize=24, fontfamily='Arial', labelpad=10)
# ax.set_title("State Visited at Each Timestep Per Trajectory (Grouped by Return)",
#              fontsize=20, fontfamily='Arial', pad=20)

# ===== LEGEND FOR STATES =====
# Mapping from state number to subscript character
subscript_map = {0: '₀', 1: '₁', 2: '₂', 3: '₃'}

legend_handles = [
    Patch( facecolor=state_colors[s], edgecolor='black', label=rf'$\mathrm{{s}}_{{{s}}}$')
    for s in states_present
]

leg = ax.legend(
    handles=legend_handles,
    title="States",
    loc='center left',
    bbox_to_anchor=(1, 0.5),  # outside on the right
    frameon=True,
    ncol=1,
    fontsize=24,        # label font size
    title_fontsize=24   # title font size
)

# set font family for title and labels
plt.setp(leg.get_title(), fontfamily='Arial')
for text in leg.get_texts():
    text.set_fontfamily('Arial')

# ===== HORIZONTAL LINES BETWEEN TIMESTEPS =====
n_rows, n_cols = decimal_states_arr.shape
for y in np.arange(0.5, n_rows, 1.0):
    ax.hlines(y, -0.5, n_cols - 0.5, colors='white', linestyles='-',
              linewidth=1, alpha=0.6)

plt.tight_layout()

plt.show()
